---
title: "数论——乘法逆元"
format:
  html:
    embed-resources: true
    toc: true
    theme: cosmo
    code-copy: true
  pdf:
    pdf-engine: xelatex
    documentclass: article
---

::: {.callout-caution}
### 📖 任务描述
请你编写一个 `Combinatorics` 类，用于在 $O(1)$ 的时间复杂度内，快速回答 $\binom{n}{k} \pmod{10^9+7}$ 的查询。

**具体要求**：
1. **初始化 (`__init__`)**：
   - 传入一个最大范围 `max_n`。
   - 预处理出两个极其重要的数组：
     - `fact[i]`：存放 $i! \pmod P$ 的结果。（阶乘数组）
     - `inv_fact[i]`：存放 $i!$ 的乘法逆元 $\pmod P$ 的结果。（阶乘逆元数组）
   *(提示：`fact[i] = (fact[i-1] * i) % P`，然后用你的 `myPow` 算出逆元)*
2. **查询方法 (`comb(n, k)`)**：
   - 如果 $k < 0$ 或 $k > n$，返回 0。
   - 否则，直接利用预处理好的 `fact` 和 `inv_fact` 数组，用 $O(1)$ 的时间通过公式算出并返回 $\binom{n}{k} \pmod P$。
:::

In [3]:
class Combinatorics:
    def __init__(self, max_n: int, mod: int = 10**9 + 7):
        self.mod = mod
        self.fact = [1] * (max_n + 1)
        self.inv_fact = [1] * (max_n + 1)
        for i in range(1, max_n + 1):  # 预处理阶乘：fact[i] = i! % mod
            self.fact[i] = (self.fact[i - 1] * i) % self.mod
        self.inv_fact[max_n] = pow(self.fact[max_n], self.mod - 2, self.mod)  # 费马小定理：(max_n!) 的逆元就是 (max_n!)^(mod-2) % mod
        for i in range(max_n - 1, -1, -1):  # 降维打击：O(N) 倒推所有逆元！避免了 N 次快速幂
            self.inv_fact[i] = (self.inv_fact[i + 1] * (i + 1)) % self.mod

    def comb(self, n: int, k: int) -> int:
        """O(1) 时间内求出组合数 C(n, k) % mod"""
        if k < 0 or k > n:
            return 0
        res = (self.fact[n] * self.inv_fact[k]) % self.mod  # 公式: n! * inv(k!) * inv((n-k)!) % mod
        res = (res * self.inv_fact[n - k]) % self.mod
        return res

::: {.callout-important}
### 💡 思路讲解

凡是涉及到概率、排列组合的困难题，往往会要求“频繁地计算 $\binom{n}{k} \pmod{10^9+7}$”，而且 $n$ 和 $k$ 可能高达 $10^5$。我们知道组合数的公式是：

$$\binom{n}{k} = \frac{n!}{k!(n-k)!}$$

在取模世界里，它就变成了：

$$\binom{n}{k} \equiv n! \times \text{inv}(k!) \times \text{inv}((n-k)!) \pmod P$$
:::

::: {.callout-note}
### 💡 复杂度分析
* **时间复杂度**：
  - 初始化：$O(N \log P)$，预处理 $N$ 个阶乘并求逆元（每次求逆元用快速幂需 $\log P$）。
  - 查询 `comb(n, k)`：$O(1)$。瞬间查表相乘！
* **空间复杂度**：$O(N)$。需要两个长度为 $N$ 的数组。
:::

## 📝 题目 1621：大小为 K 的不重叠线段的数目

::: {.callout-caution}
### 📖 题目描述
给你 $n$ 个点（编号从 $0$ 到 $n-1$）。请你在这些点上画 $k$ 条线段。
**规则：**
1. 线段不能重叠。
2. 但是！相邻的线段**可以共享同一个端点**（例如，线段 [0,1] 和 [1,2] 是合法的）。
3. 线段的长度必须大于 0。

请你返回画这 $k$ 条线段的方案数，结果对 $10^9 + 7$ 取模。

**输入输出示例**：

* **示例 1**：
    * **输入**：`n = 4, k = 2`
    * **输出**：`5`
    * **解释**：可以在以下点之间画线段：
      - [0,1] 和 [1,2] （共享端点 1）
      - [0,1] 和 [1,3] （共享端点 1）
      - [0,1] 和 [2,3]
      - [0,2] 和 [2,3] （共享端点 2）
      - [1,2] 和 [2,3] （共享端点 2）
:::

In [4]:
class Solution1621:
    def numberOfSets(self, n: int, k: int) -> int:
        math_tool = Combinatorics(n + k, 10**9 + 7)  # N个点加 K-1 个虚拟点，最多需要算组合数 N+K
        return math_tool.comb(n + k - 1, 2 * k)

In [6]:
#| code-fold: true
def test_1621(func):
    test_cases = [
        {"desc": "示例 1: 常规小数据", "n": 4, "k": 2, "expected": 5},
        {"desc": "示例 2: 只有一条线段", "n": 3, "k": 1, "expected": 3},
        {"desc": "示例 3: 稍大数字", "n": 30, "k": 7, "expected": 796297179},
        {"desc": "边界: 刚好能铺满 (n个点连成n-1条线)", "n": 3, "k": 2, "expected": 1},
        {"desc": "边界: 最短的单线段", "n": 2, "k": 1, "expected": 1},
        {"desc": "不可能任务: 点太少画不出k条线", "n": 2, "k": 2, "expected": 0},
        {"desc": "不可能任务: 点为1", "n": 1, "k": 1, "expected": 0},
        {"desc": "大厂极限压测: N和K都在中等偏上", "n": 1000, "k": 200, "expected": 60572379} # 考验取模和逆元是否溢出
    ]

    passed = 0
    print(f"{'ID':<3} | {'结果':<6} | {'预期':<10} | {'实际':<10} | {'描述'}")
    print("-" * 65)

    for i, tc in enumerate(test_cases):
        actual = func(tc["n"], tc["k"])
        is_pass = actual == tc["expected"]
        status = "✅ PASS" if is_pass else "❌ FAIL"
        if is_pass:
            passed += 1

        print(f"{i+1:<3} | {status:<6} | {tc['expected']:<10} | {actual:<10} | {tc['desc']}")

    print("-" * 65)
    print(f"测试总结: 通过 {passed}/{len(test_cases)}")

# 运行测试
test_1621(Solution1621().numberOfSets)

ID  | 结果     | 预期         | 实际         | 描述
-----------------------------------------------------------------
1   | ✅ PASS | 5          | 5          | 示例 1: 常规小数据
2   | ✅ PASS | 3          | 3          | 示例 2: 只有一条线段
3   | ✅ PASS | 796297179  | 796297179  | 示例 3: 稍大数字
4   | ✅ PASS | 1          | 1          | 边界: 刚好能铺满 (n个点连成n-1条线)
5   | ✅ PASS | 1          | 1          | 边界: 最短的单线段
6   | ✅ PASS | 0          | 0          | 不可能任务: 点太少画不出k条线
7   | ✅ PASS | 0          | 0          | 不可能任务: 点为1
8   | ✅ PASS | 60572379   | 60572379   | 大厂极限压测: N和K都在中等偏上
-----------------------------------------------------------------
测试总结: 通过 8/8


::: {.callout-important}
### 💡 思路讲解

**核心推导过程：**

1. **纯粹的非重叠挑选**：
   在 $n$ 个点上画 $k$ 条线段，本质上需要我们选出 $2k$ 个端点。
   如果题目严格规定“任何端点都绝对不能重叠”，那答案非常简单：直接在 $n$ 个点里挑 $2k$ 个点，即 $\binom{n}{2k}$。

2. **处理“允许端点重叠”的棘手条件**：
   题目允许相邻线段共享端点（比如 `[0,1]` 和 `[1,2]` 共享了点 `1`）。这意味着，在挑选的 $2k$ 个位置里，有些点被重复选中了，直接用组合数会漏算或者算错。

3. **魔法降临：强行拉开并塞入“虚拟节点”**：
   为了消除这种“重叠”，我们想象用一双无形的手，在每一处可能重合的线段交界处，**强行塞入一个虚拟的点**，把它们硬生生隔开！
   - 因为有 $k$ 条线段，它们之间最多有 $k-1$ 个交界处。
   - 所以，我们最多需要向原宇宙中塞入 $k-1$ 个虚拟点。
   - 塞完之后，这个新宇宙的总点数膨胀成了：$n + k - 1$ 个。

4. **在新宇宙中降维打击**：
   在这个充满虚拟点的新宇宙里，**线段端点再也绝对不可能重合了！** 我们只需要在这个全新的 $n + k - 1$ 个点中，老老实实地选出 $2k$ 个绝对不同的端点即可。

**终极公式**：方案数等于 $\binom{n + k - 1}{2k}$。
:::

::: {.callout-note}
### 💡 复杂度分析
* **时间复杂度**：$O(N + K)$。
  - 初始化 `Combinatorics` 兵工厂（预处理阶乘和逆元）需要 $O(N + K)$ 的时间。
  - 调用 `comb(n + k - 1, 2 * k)` 查表计算公式，只需要 $O(1)$ 的极致速度！
* **空间复杂度**：$O(N + K)$。
  - 需要在兵工厂内部开辟两个长度为 $N + K$ 的一维数组，分别存储阶乘 (`fact`) 和阶乘的乘法逆元 (`inv_fact`)。
:::

## 📝 题目 2400：恰好移动 k 步到达某一位置的方法数目

::: {.callout-caution}
### 📖 题目描述
给你两个 **正** 整数 `startPos` 和 `endPos` 。最初，你站在无限数轴上的 `startPos` 处。
在一步移动中，你可以向左或者向右移动一个位置。

给你一个正整数 `k` ，请你返回从 `startPos` 出发，**恰好移动 `k` 步** 并到达 `endPos` 的 **不同** 方法数目。
由于答案可能会很大，请返回对 $10^9 + 7$ 取余 后的结果。

**输入输出示例**：

* **示例 1**：
    * **输入**：`startPos = 1`, `endPos = 2`, `k = 3`
    * **输出**：`3`
    * **解释**：存在 3 种从 1 到 2 且恰好移动 3 步的方法：
      - 1 -> 2 -> 3 -> 2 (右，右，左)
      - 1 -> 2 -> 1 -> 2 (右，左，右)
      - 1 -> 0 -> 1 -> 2 (左，右，右)

* **示例 2**：
    * **输入**：`startPos = 2`, `endPos = 5`, `k = 10`
    * **输出**：`0`
    * **解释**：不可能经过 10 步跑到 5。比如全往右走也只能走到 12，全往左走到 -8。不管怎么走都到不了 5？不对，示例 2 的意思是，距离是 3，走 10 步到不了（因为 10 步走出来的净位移只能是偶数！）。
:::

In [7]:
class Solution2400:
    def numberOfWays(self, startPos: int, endPos: int, k: int) -> int:
        d = abs(startPos - endPos)
        if d > k or (d + k) & 1:
            return 0
        R = (d + k) >> 1
        return Combinatorics(k).comb(k, R)

In [10]:
#| code-fold: true
def test_2400(func):
    test_cases = [
        {"desc": "示例 1: 常规步数", "startPos": 1, "endPos": 2, "k": 3, "expected": 3},
        {"desc": "示例 2: 奇偶性不符，绝对到不了", "startPos": 2, "endPos": 5, "k": 10, "expected": 0},
        {"desc": "距离奇偶拦截 (距离1，走2步)", "startPos": 1, "endPos": 2, "k": 2, "expected": 0},
        {"desc": "全速冲刺刚好到达 (距离5，走5步)", "startPos": 0, "endPos": 5, "k": 5, "expected": 1},
        {"desc": "原地踏步，偶数步 (向左向右各一半)", "startPos": 0, "endPos": 0, "k": 4, "expected": 6}, # C(4, 2)
        {"desc": "原地踏步，奇数步", "startPos": 0, "endPos": 0, "k": 3, "expected": 0},
        {"desc": "距离过大拦截 (距离6，走5步)", "startPos": 0, "endPos": 6, "k": 5, "expected": 0},
        {"desc": "大厂极限压测 (模数挑战)", "startPos": 1, "endPos": 10, "k": 1001, "expected": 440462164} # 需要大数取模！
    ]

    passed = 0
    print(f"{'ID':<3} | {'结果':<6} | {'预期':<10} | {'实际':<10} | {'描述'}")
    print("-" * 75)

    for i, tc in enumerate(test_cases):
        actual = func(tc["startPos"], tc["endPos"], tc["k"])
        is_pass = actual == tc["expected"]
        status = "✅ PASS" if is_pass else "❌ FAIL"
        if is_pass:
            passed += 1

        print(f"{i+1:<3} | {status:<6} | {tc['expected']:<10} | {actual:<10} | {tc['desc']}")

    print("-" * 75)
    print(f"测试总结: 通过 {passed}/{len(test_cases)}")

# 运行测试
test_2400(Solution2400().numberOfWays)

ID  | 结果     | 预期         | 实际         | 描述
---------------------------------------------------------------------------
1   | ✅ PASS | 3          | 3          | 示例 1: 常规步数
2   | ✅ PASS | 0          | 0          | 示例 2: 奇偶性不符，绝对到不了
3   | ✅ PASS | 0          | 0          | 距离奇偶拦截 (距离1，走2步)
4   | ✅ PASS | 1          | 1          | 全速冲刺刚好到达 (距离5，走5步)
5   | ✅ PASS | 6          | 6          | 原地踏步，偶数步 (向左向右各一半)
6   | ✅ PASS | 0          | 0          | 原地踏步，奇数步
7   | ✅ PASS | 0          | 0          | 距离过大拦截 (距离6，走5步)
8   | ✅ PASS | 440462164  | 440462164  | 大厂极限压测 (模数挑战)
---------------------------------------------------------------------------
测试总结: 通过 8/8


::: {.callout-important}
### 💡 思路讲解

遇到这种求方案数的题，千万别急着写递归！我们先用数学把它的底牌看穿。

**核心推导：**

1. 假设我们一共向**右**走了 $R$ 步，向**左**走了 $L$ 步。
2. 题目规定了总步数必须是 $k$，所以：
   $$R + L = k$$
3. 题目规定了最终的物理位移必须是距离差。设 $d = |endPos - startPos|$，那么：
   $$R - L = d$$
4. 我们把这两个方程相加：
   $$(R + L) + (R - L) = k + d$$
   $$2R = k + d$$
   所以，必须向右走的步数 $R = \frac{k + d}{2}$。

**终极物理审判（拦截非法情况）：**
- 如果 $d > k$：连全速冲刺都跑不到终点，直接返回 0。
- 如果 $k + d$ 不是偶数：意味着 $R$ 算出来是个小数！物理上不可能走“半步”，说明永远走不到，直接返回 0。（这就是为什么走 10 步到不了距离 3 的终点）。

**组合数学降临：**
排除了不可能的情况后，剩下的就是纯纯的排列组合了：
我们一共要走 $k$ 步，其中有 $R$ 步必须分配给“向右走”，剩下的自然是向左走。
方案数不就是在 $k$ 个位置里，挑出 $R$ 个位置填入“右”吗？
**答案就是：$\binom{k}{R}$**！
:::

::: {.callout-note}
### 💡 复杂度分析
* **时间复杂度**：$O(k)$。
  算法的核心耗时在于实例化 `Combinatorics(k)`。预处理 $k$ 的阶乘和逆元需要 $O(k)$ 的时间。最后的 `comb` 查询只需 $O(1)$。
* **空间复杂度**：$O(k)$。
  `Combinatorics` 类在内部开辟了长度为 $k+1$ 的 `fact` 和 `inv_fact` 数组。
:::